# Stage 6 — LoRA Fine-tuning of IDM-VTON on VITON-HD

**What Stage 5 did:** ran IDM-VTON *pre-trained* inference — good fit, but color can drift because our custom inputs differ from the VITON-HD distribution the model was trained on.

**Stage 6 goal:** fine-tune *only the attention layers* of TryonUNet via [LoRA](https://arxiv.org/abs/2106.09685) on the VITON-HD **training** split so the model better handles our specific person + lighting.

### Why LoRA, not full fine-tuning?
Full fine-tuning would need ~3× the inference VRAM (gradients + optimizer states for all 12 GB of TryonUNet weights). LoRA adds tiny rank-4 adapter matrices to the attention projections only (~2 M trainable params vs 860 M total) so the optimizer state is negligible and we stay within the T4's 15.6 GB.

### Memory layout during training (T4)
| Component | VRAM | Gradient? |
|-----------|------|-----------|
| TryonUNet backbone (fp16) | 6.3 GB | frozen |
| LoRA adapters (fp16) | ~50 MB | **yes** |
| GarmentUNet (fp16) | 5.2 GB | frozen |
| VAE + CLIP (fp16) | 1.2 GB | frozen |
| Activations (grad ckpt, bs=1) | ~1.5 GB | — |
| Optimizer (AdamW, LoRA only) | ~100 MB | — |
| **Total** | **~14.4 GB** | |

### Run order
1. Cell 1 — GPU check
2. Cell 3 — Install *(kills runtime → expected)*
3. Cell 4 — Verify
4. Cells 6-7 — Mount Drive, set paths
5. Cells 9-10 — Download VITON-HD, find train split
6. Cell 12 — Compat patches (re-run on every restart)
7. Cell 14 — Load frozen models + encode prompts
8. Cells 16-17 — Load TryonUNet + apply LoRA
9. Cells 19-20 — Dataset + preview
10. Cells 22-24 — Optimizer, training step, training loop
11. Cell 26 — Plot loss curve
12. Cell 28 — Quick inference test with trained LoRA

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU — Runtime → Change runtime type → T4 GPU')
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU  : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {vram:.1f} GB')
assert vram >= 14, f'Need >= 14 GB VRAM, got {vram:.1f} GB'
print('OK')


## 1. Install Dependencies

Same pins as Stage 5. Skip this cell if you already ran it in this session.
The cell calls `os.kill` at the end — Colab will say **"session crashed"**, that is expected. Re-run from Cell 4.

In [ ]:
import subprocess, sys, os
packages = [
    'huggingface_hub==0.22.2', 'diffusers==0.25.0', 'transformers==4.40.0',
    'accelerate==0.30.0', 'peft==0.10.0', 'safetensors',
    'xformers', 'kagglehub', 'triton',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade'] + packages)
print('Done — restarting...')
os.kill(os.getpid(), 9)


In [ ]:
# ── START HERE after runtime restarts ───────────────────────────────────────
import huggingface_hub, diffusers, transformers, accelerate, peft
print(f'huggingface_hub : {huggingface_hub.__version__}')
print(f'diffusers       : {diffusers.__version__}')
print(f'transformers    : {transformers.__version__}')
print(f'accelerate      : {accelerate.__version__}')
print(f'peft            : {peft.__version__}')
print('All OK')


## 2. Mount Drive + Set Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
DRIVE      = '/content/drive/MyDrive/VirtualTryOn'
INPUT_DIR  = f'{DRIVE}/input_images'
OUTPUT_DIR = f'{DRIVE}/output_images'
CKPT_DIR   = f'{DRIVE}/checkpoints'
LORA_DIR   = f'{CKPT_DIR}/stage6_lora'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LORA_DIR,   exist_ok=True)
print('Paths set.')
print(f'  LoRA weights will save to: {LORA_DIR}')


## 3. Download VITON-HD (Training Split)

This time we use the **train** split (~14,000 pairs) instead of the test split.
Dataset is not saved to Drive — re-download each session.

In [ ]:
import os
os.environ['KAGGLE_TOKEN'] = 'KGAT_fbc21b5329e8cbe6a20395ed9e25c159'
import kagglehub

print('Downloading VITON-HD (~3-5 min)...')
dataset_path = kagglehub.dataset_download(
    'marquis03/high-resolution-viton-zalando-dataset')
print(f'Downloaded to: {dataset_path}')

# Find train split (has 'image' subfolder + is named 'train')
train_base = None
for root, dirs, _ in os.walk(dataset_path):
    if os.path.basename(root) == 'train' and 'image' in dirs:
        train_base = root; break
if train_base is None:
    # Some versions have it directly under dataset_path
    for item in os.listdir(dataset_path):
        p = os.path.join(dataset_path, item)
        if os.path.isdir(p) and 'train' in item.lower():
            train_base = p; break
assert train_base, 'Could not find train split — check dataset_path manually'
print(f'Train split: {train_base}')
print('Sub-folders:', sorted(os.listdir(train_base)))


In [ ]:
# Count pairs and show folder structure
img_dir = os.path.join(train_base, 'image')
n_pairs = len(os.listdir(img_dir))
print(f'Training pairs: {n_pairs}')
print()
for folder in sorted(os.listdir(train_base)):
    fp = os.path.join(train_base, folder)
    if os.path.isdir(fp):
        files = os.listdir(fp)
        print(f'  {folder:35s} {len(files):6d} files')


## 4. Compatibility Patches + Session Restore

**Run on every kernel restart.** Identical to Stage 5's restore cell.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# RESTORE SESSION — run on every kernel restart
# ════════════════════════════════════════════════════════════════════════════
import os, sys, gc, torch, torch.nn as nn

model_path = '/content/idm-vton-weights'
DTYPE, DEVICE = torch.float16, 'cuda'
TARGET_W, TARGET_H = 768, 1024

assert os.path.isdir(model_path), 'Weights missing — run Stage 5 Cell 14 first'
if '/content/IDM-VTON' not in sys.path:
    sys.path.insert(0, '/content/IDM-VTON')

import diffusers.models.embeddings as _emb
if not hasattr(_emb, 'PositionNet'):
    _emb.PositionNet = type('PositionNet', (nn.Module,), {
        '__init__': lambda self, *a, **kw: nn.Module.__init__(self),
        'forward':  lambda self, x: x,
    })
    print('Patched: PositionNet')

for _f in ['/content/IDM-VTON/src/unet_hacked_tryon.py',
           '/content/IDM-VTON/src/unet_hacked_garmnet.py']:
    _s = open(_f).read()
    _p = _s.replace(', _remove_lora=_remove_lora', '')
    if _p != _s: open(_f, 'w').write(_p); print(f'Patched: _remove_lora in {os.path.basename(_f)}')

import accelerate.utils.memory as _acc
if not hasattr(_acc, 'clear_device_cache'):
    def _cdc(gc_=False):
        if gc_: gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    _acc.clear_device_cache = _cdc; print('Patched: clear_device_cache')

import huggingface_hub.errors as _hfe
for _n, _b in [('HFValidationError', ValueError), ('LocalEntryNotFoundError', FileNotFoundError),
               ('EntryNotFoundError', FileNotFoundError), ('RepositoryNotFoundError', FileNotFoundError),
               ('HfHubHTTPError', OSError)]:
    if not hasattr(_hfe, _n): setattr(_hfe, _n, type(_n, (_b,), {})); print(f'Patched: {_n}')

import transformers as _t
if not hasattr(_t, 'EncoderDecoderCache'):
    from transformers import DynamicCache
    _EDC = type('EncoderDecoderCache', (DynamicCache,), {})
    _t.EncoderDecoderCache = _EDC
    sys.modules['transformers'].EncoderDecoderCache = _EDC
    print('Patched: EncoderDecoderCache')

for k in [k for k in list(sys.modules) if 'peft' in k]: del sys.modules[k]

from src.tryon_pipeline import StableDiffusionXLInpaintPipeline as TryonPipeline
from src.unet_hacked_garmnet import UNet2DConditionModel as GarmentUNet
from src.unet_hacked_tryon   import UNet2DConditionModel as TryonUNet
print('Session restored')


## 5. Load Frozen Models + Encode Training Prompts

VAE, GarmentUNet, and CLIP are loaded with `requires_grad_(False)` — they never receive gradients.  
Text encoders are loaded just long enough to encode the two training prompts, then freed to recover ~1.6 GB VRAM before the training loop starts.

In [ ]:
import gc, warnings, torch
from accelerate import init_empty_weights
from safetensors import safe_open
from diffusers import DDPMScheduler, AutoencoderKL
from transformers import (CLIPImageProcessor, CLIPVisionModelWithProjection,
                          CLIPTextModel, CLIPTextModelWithProjection, CLIPTokenizer)

warnings.filterwarnings('ignore', category=FutureWarning)

# ── DDPM scheduler (used for adding noise during training) ───────────────
noise_scheduler = DDPMScheduler.from_pretrained(model_path, subfolder='scheduler')
print('DDPM scheduler loaded')

# ── VAE ──────────────────────────────────────────────────────────────────
vae = AutoencoderKL.from_pretrained(
    '/content/sdxl-vae-fp16', torch_dtype=torch.float16).to(DEVICE)
vae.requires_grad_(False)
print(f'VAE loaded.  VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB')

# ── GarmentUNet (frozen garment feature extractor) ───────────────────────
print('Loading GarmentUNet...')
_sd_enc = {}
with safe_open(f'{model_path}/unet_encoder/diffusion_pytorch_model.safetensors',
               framework='pt', device=DEVICE) as f:
    for k in f.keys(): _sd_enc[k] = f.get_tensor(k).half()
gc.collect(); torch.cuda.empty_cache()
with init_empty_weights():
    unet_encoder = GarmentUNet.from_config(f'{model_path}/unet_encoder')
unet_encoder.load_state_dict(_sd_enc, assign=True, strict=False)
unet_encoder.requires_grad_(False)
del _sd_enc; gc.collect(); torch.cuda.empty_cache()
print(f'GarmentUNet loaded.  VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB')

# ── CLIP image encoder ───────────────────────────────────────────────────
image_encoder = CLIPVisionModelWithProjection.from_pretrained(
    model_path, subfolder='image_encoder',
    torch_dtype=torch.float16, low_cpu_mem_usage=True).to(DEVICE)
image_encoder.requires_grad_(False)
feature_extractor = CLIPImageProcessor()
print(f'CLIP loaded.  VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB')

# ── Text encoders: load temporarily, encode prompts, then free ───────────
import os, shutil
from huggingface_hub import hf_hub_download

def _recover_tokenizer(subfolder):
    tok_dir = f'{model_path}/{subfolder}'
    os.makedirs(tok_dir, exist_ok=True)
    for fname in ['vocab.json', 'merges.txt', 'tokenizer_config.json', 'special_tokens_map.json']:
        dest = os.path.join(tok_dir, fname)
        if not os.path.exists(dest):
            tmp = hf_hub_download(repo_id='yisol/IDM-VTON',
                                  filename=f'{subfolder}/{fname}',
                                  local_dir='/tmp/tok_recover')
            shutil.copy(tmp, dest)
            print(f'  Recovered {subfolder}/{fname}')

_recover_tokenizer('tokenizer'); _recover_tokenizer('tokenizer_2')

tokenizer   = CLIPTokenizer.from_pretrained(model_path, subfolder='tokenizer')
tokenizer_2 = CLIPTokenizer.from_pretrained(model_path, subfolder='tokenizer_2')
text_encoder = CLIPTextModel.from_pretrained(
    model_path, subfolder='text_encoder',
    torch_dtype=torch.float16, low_cpu_mem_usage=True).to(DEVICE)
text_encoder_2 = CLIPTextModelWithProjection.from_pretrained(
    model_path, subfolder='text_encoder_2',
    torch_dtype=torch.float16, low_cpu_mem_usage=True).to(DEVICE)

# Build a pipe stub just to use encode_prompt helper
pipe_stub = TryonPipeline(
    vae=vae, text_encoder=text_encoder, text_encoder_2=text_encoder_2,
    tokenizer=tokenizer, tokenizer_2=tokenizer_2,
    unet=None, unet_encoder=unet_encoder,
    scheduler=noise_scheduler, image_encoder=image_encoder,
    feature_extractor=feature_extractor,
)

# Prompt for denoising UNet (describes the target person + garment)
TRAIN_PROMPT     = 'a person wearing a t-shirt, photorealistic, high quality'
TRAIN_NEG_PROMPT = 'monochrome, lowres, bad anatomy, worst quality, low quality'
# Prompt for garment encoder (describes the clothing item)
CLOTH_PROMPT = 'a flat-lay photograph of a t-shirt on white background'

with torch.no_grad():
    prompt_embeds, _, pooled_embeds, _ = pipe_stub.encode_prompt(
        TRAIN_PROMPT, num_images_per_prompt=1,
        do_classifier_free_guidance=False)
    cloth_prompt_embeds, _, cloth_pooled, _ = pipe_stub.encode_prompt(
        CLOTH_PROMPT, num_images_per_prompt=1,
        do_classifier_free_guidance=False)

# Free text encoders — they're no longer needed
pipe_stub.text_encoder = pipe_stub.text_encoder_2 = None
del text_encoder, text_encoder_2, tokenizer, tokenizer_2
gc.collect(); torch.cuda.empty_cache()
print(f'Prompts encoded, text encoders freed.  VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB')
print(f'  prompt_embeds : {prompt_embeds.shape}')
print(f'  pooled_embeds : {pooled_embeds.shape}')


## 6. Load TryonUNet + Apply LoRA

We first read the UNet's `config.json` to find `in_channels` — IDM-VTON uses 13 (noisy 4 + mask 1 + agnostic 4 + pose 4) rather than the standard SDXL inpainting 9.  

Then we apply `peft.LoraConfig` to the four attention projection layers in every transformer block: `to_q`, `to_k`, `to_v`, `to_out.0`.  

Finally `enable_gradient_checkpointing()` recomputes activations during the backward pass instead of storing them — trades ~30% extra compute for ~40% less activation memory.

In [ ]:
import json

with open(f'{model_path}/unet/config.json') as f:
    unet_cfg = json.load(f)

in_channels = unet_cfg.get('in_channels', 9)
print(f'TryonUNet in_channels : {in_channels}')
print(f'  9  = standard SDXL inpainting (noisy + mask + agnostic)')
print(f'  13 = IDM-VTON extended       (noisy + mask + agnostic + pose)')
print(f'Sample size : {unet_cfg.get("sample_size")}')
print(f'Block types : {unet_cfg.get("down_block_types")}')


In [ ]:
from accelerate import init_empty_weights
from peft import LoraConfig, get_peft_model
import gc, torch

LORA_RANK  = 4
LORA_ALPHA = 4

# Load TryonUNet directly to VRAM (avoids CPU RAM OOM — same trick as Stage 5)
print('Loading TryonUNet (disk -> VRAM fp32 -> fp16) ~2-3 min...')
_sd = torch.load(f'{model_path}/unet/diffusion_pytorch_model.bin', map_location=DEVICE)
for k in list(_sd): _sd[k] = _sd[k].half()
gc.collect(); torch.cuda.empty_cache()
with init_empty_weights():
    unet = TryonUNet.from_config(f'{model_path}/unet')
unet.load_state_dict(_sd, assign=True, strict=False)
del _sd; gc.collect(); torch.cuda.empty_cache()
print(f'TryonUNet loaded.  VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB')

# Freeze the backbone — only LoRA adapters will be trained
unet.requires_grad_(False)

# Apply LoRA to attention projections in every transformer block
lora_cfg = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=['to_q', 'to_k', 'to_v', 'to_out.0'],
    lora_dropout=0.1,
    bias='none',
)
unet = get_peft_model(unet, lora_cfg)
unet.print_trainable_parameters()  # should be ~0.2% of total

# Gradient checkpointing: recompute activations on backward instead of storing
unet.enable_gradient_checkpointing()

vram_total = torch.cuda.memory_allocated() / 1e9
print(f'All models loaded with LoRA.  VRAM: {vram_total:.2f} / 15.6 GB')
print(f'Headroom for training activations: {15.6 - vram_total:.2f} GB')


## 7. Dataset

The `VITONHDDataset` class discovers the subfolder names automatically (they vary between dataset versions) and loads 5 images per sample:

| Key | Description | Normalization |
|-----|-------------|---------------|
| `person` | Ground-truth person wearing the clothing | [-1, 1] |
| `cloth` | Flat-lay clothing on white background | [-1, 1] |
| `agn` | Person with clothing region erased | [-1, 1] |
| `mask` | Binary mask of clothing region | [0, 1] |
| `pose` | OpenPose skeleton overlay | [-1, 1] |

`person` is the **training target** — the model learns to reconstruct it by inpainting the masked region conditioned on `cloth`, `agn`, and `pose`.

In [ ]:
import os
import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader

class VITONHDDataset(Dataset):
    """VITON-HD dataset for IDM-VTON LoRA fine-tuning."""

    def __init__(self, split_root, target_w=768, target_h=1024, max_samples=None):
        self.root = split_root
        self.W, self.H = target_w, target_h

        img_dir = os.path.join(split_root, 'image')
        self.stems = sorted(os.path.splitext(f)[0] for f in os.listdir(img_dir))
        if max_samples:
            self.stems = self.stems[:max_samples]

        # Auto-discover subfolder names (vary by dataset version)
        subdirs = os.listdir(split_root)
        def _find(kw, excl=None):
            for d in subdirs:
                dl = d.lower()
                if kw in dl and (excl is None or excl not in dl):
                    return os.path.join(split_root, d)
            return None

        self.dirs = {
            'person': os.path.join(split_root, 'image'),
            'cloth':  os.path.join(split_root, 'cloth'),
            'agn':    _find('agnostic', 'mask'),
            'mask':   _find('agnostic-mask') or _find('agnostic_mask'),
            'pose':   _find('openpose', 'json'),
        }
        print(f'Dataset: {len(self.stems)} pairs')
        for k, v in self.dirs.items():
            status = 'OK' if v and os.path.isdir(v) else 'MISSING (fallback)'
            print(f'  {k:8s}: {status}  ({v})')

    def __len__(self): return len(self.stems)

    def _load(self, directory, stem, mode='RGB'):
        if directory is None or not os.path.isdir(directory):
            return None
        for ext in ['.jpg', '.jpeg', '.png']:
            p = os.path.join(directory, stem + ext)
            if os.path.exists(p):
                return Image.open(p).convert(mode).resize(
                    (self.W, self.H), Image.BILINEAR)
        return None

    @staticmethod
    def _t(img, norm=True):
        """PIL -> (C,H,W) float32 tensor."""
        arr = np.array(img).astype(np.float32)
        arr = (arr / 127.5 - 1.0) if norm else (arr / 255.0)
        if arr.ndim == 2: arr = arr[..., None]
        return torch.from_numpy(arr).permute(2, 0, 1)

    def __getitem__(self, idx):
        s = self.stems[idx]
        W, H = self.W, self.H
        blank_rgb = Image.new('RGB', (W, H))
        blank_l   = Image.new('L',   (W, H), 128)

        person = self._load(self.dirs['person'], s) or blank_rgb
        cloth  = self._load(self.dirs['cloth'],  s) or blank_rgb
        agn    = self._load(self.dirs['agn'],    s) or person
        mask   = self._load(self.dirs['mask'],   s, 'L') or blank_l
        pose   = self._load(self.dirs['pose'],   s) or person

        return {
            'person': self._t(person),       # (3,H,W) target
            'cloth':  self._t(cloth),         # (3,H,W) garment
            'agn':    self._t(agn),           # (3,H,W) agnostic
            'mask':   self._t(mask, norm=False),  # (1,H,W) [0,1]
            'pose':   self._t(pose),          # (3,H,W) skeleton
        }


In [ ]:
import matplotlib.pyplot as plt

# Use max_samples to limit training time on free Colab
# 2000 samples * ~1.5s/step = ~50 min for one pass
# Set max_samples=None to use the full ~14k training set
MAX_SAMPLES = 2000

train_dataset = VITONHDDataset(train_base, max_samples=MAX_SAMPLES)

# Preview one sample
sample = train_dataset[0]
def to_pil(t):
    arr = t.permute(1,2,0).numpy()
    arr = ((arr + 1) * 127.5).clip(0,255).astype('uint8') if arr.min() < 0 \
          else (arr * 255).clip(0,255).astype('uint8')
    if arr.shape[2] == 1: arr = arr[:,:,0]
    return Image.fromarray(arr)

fig, axes = plt.subplots(1, 5, figsize=(20, 5))
for ax, key, cmap in zip(axes,
        ['person','cloth','agn','mask','pose'],
        [None, None, None, 'gray', None]):
    ax.imshow(to_pil(sample[key]), cmap=cmap)
    ax.set_title(key); ax.axis('off')
plt.suptitle('Training sample preview'); plt.tight_layout(); plt.show()


## 8. Training

Three cells:
- **Cell 22** — optimizer, SDXL helpers, hyperparameters
- **Cell 23** — `training_step()` function (one denoising step)
- **Cell 24** — main loop with gradient accumulation + checkpointing

### How the denoising loss works
1. Encode the **ground-truth person** (wearing the target clothing) to VAE latents
2. Sample a random noise level `t ∈ [0, 1000]` and add noise → noisy latents
3. Encode `cloth` latents and pass through frozen **GarmentUNet** → garment feature residuals
4. Concatenate noisy + mask + agnostic (+ pose if `in_channels=13`) → UNet input
5. TryonUNet (LoRA) predicts the added noise
6. **MSE loss** between predicted noise and actual noise

The model learns: *given an agnostic person + garment features, reconstruct the clothing region.*

In [ ]:
import torch, gc
import torch.nn.functional as F

# ── Hyperparameters ───────────────────────────────────────────────────────
LR           = 1e-4    # learning rate for LoRA adapters
MAX_STEPS    = 5000    # total gradient-update steps
GRAD_ACCUM   = 4       # accumulate over 4 batches before updating (effective bs=4)
SAVE_EVERY   = 500     # save LoRA checkpoint every N steps
LOG_EVERY    = 50      # print loss every N steps

# ── Optimizer (AdamW on LoRA params only) ─────────────────────────────────
lora_params = [p for p in unet.parameters() if p.requires_grad]
print(f'Trainable parameters: {sum(p.numel() for p in lora_params):,}')
optimizer = torch.optim.AdamW(
    lora_params, lr=LR, betas=(0.9, 0.999), eps=1e-8, weight_decay=1e-2)
scaler = torch.cuda.amp.GradScaler()

# ── SDXL added-time-ids helper ────────────────────────────────────────────
# SDXL conditions the UNet on original resolution + crop coords + target size.
# We use a fixed value for all training samples.
add_time_ids = pipe_stub._get_add_time_ids(
    (TARGET_H, TARGET_W), (0, 0), (TARGET_H, TARGET_W),
    dtype=torch.float16,
).to(DEVICE)

# Pre-move prompt embeddings to GPU
prompt_embeds_g  = prompt_embeds.to(DEVICE)
pooled_embeds_g  = pooled_embeds.to(DEVICE)
cloth_embeds_g   = cloth_prompt_embeds.to(DEVICE)
cloth_pooled_g   = cloth_pooled.to(DEVICE)

added_cond       = {'text_embeds': pooled_embeds_g, 'time_ids': add_time_ids}
added_cond_cloth = {'text_embeds': cloth_pooled_g,  'time_ids': add_time_ids}

print('Optimizer ready.')


In [ ]:
def training_step(batch):
    """
    One denoising training step.
    Returns scalar loss (not yet divided by GRAD_ACCUM).
    """
    # Move batch to GPU as fp16
    p  = batch['person'].unsqueeze(0).half().to(DEVICE)  # (1,3,H,W) target
    cl = batch['cloth'].unsqueeze(0).half().to(DEVICE)   # (1,3,H,W) garment
    ag = batch['agn'].unsqueeze(0).half().to(DEVICE)     # (1,3,H,W) agnostic
    mk = batch['mask'].unsqueeze(0).half().to(DEVICE)    # (1,1,H,W) [0,1]
    po = batch['pose'].unsqueeze(0).half().to(DEVICE)    # (1,3,H,W) pose

    with torch.no_grad():
        sf = vae.config.scaling_factor

        # Encode target person and conditioning images to latent space
        tgt_lat = vae.encode(p).latent_dist.sample()  * sf  # ground truth
        agn_lat = vae.encode(ag).latent_dist.sample() * sf  # agnostic cond
        cl_lat  = vae.encode(cl).latent_dist.sample() * sf  # cloth for garm enc

        # Downsample mask to latent resolution (H/8, W/8)
        lH, lW = tgt_lat.shape[2], tgt_lat.shape[3]
        mk_lat = F.interpolate(mk, size=(lH, lW), mode='nearest')

        # Sample random timestep and add noise to the target latents
        noise = torch.randn_like(tgt_lat)
        t = torch.randint(
            0, noise_scheduler.config.num_train_timesteps, (1,), device=DEVICE)
        noisy = noise_scheduler.add_noise(tgt_lat, noise, t)

        # Get garment feature residuals from frozen GarmentUNet
        down_res, mid_res = unet_encoder(
            cl_lat, t,
            encoder_hidden_states=cloth_embeds_g,
            added_cond_kwargs=added_cond_cloth,
            return_dict=False,
        )

    # Build UNet latent model input based on in_channels
    # in_channels=9 : [noisy(4) | mask(1) | agnostic(4)]
    # in_channels=13: [noisy(4) | mask(1) | agnostic(4) | pose(4)]
    if in_channels >= 13:
        with torch.no_grad():
            pose_lat = vae.encode(po).latent_dist.sample() * sf
        lmi = torch.cat([noisy, mk_lat, agn_lat, pose_lat], dim=1)
    else:
        lmi = torch.cat([noisy, mk_lat, agn_lat], dim=1)

    # Forward through LoRA-equipped TryonUNet with fp16 autocast
    with torch.cuda.amp.autocast(dtype=torch.float16):
        noise_pred = unet(
            lmi, t,
            encoder_hidden_states=prompt_embeds_g,
            added_cond_kwargs=added_cond,
            down_block_additional_residuals=[r.half() for r in down_res],
            mid_block_additional_residual=mid_res.half(),
            return_dict=False,
        )[0]

    # Standard denoising score-matching loss
    return F.mse_loss(noise_pred.float(), noise.float())


In [ ]:
import time, os

dataloader = DataLoader(
    train_dataset, batch_size=1, shuffle=True,
    num_workers=2, pin_memory=False, drop_last=True,
)
data_iter   = iter(dataloader)
loss_history = []
global_step  = 0
optimizer.zero_grad()
t0 = time.time()

print(f'Starting training: {MAX_STEPS} steps, grad_accum={GRAD_ACCUM}')
print(f'Effective batch size: {GRAD_ACCUM}')
print(f'LoRA checkpoints -> {LORA_DIR}')
print()

while global_step < MAX_STEPS:
    # Cycle through the dataloader
    try:
        batch = next(data_iter)
    except StopIteration:
        data_iter = iter(dataloader)
        batch = next(data_iter)

    # Compute loss, scale for gradient accumulation
    loss = training_step(batch) / GRAD_ACCUM
    scaler.scale(loss).backward()
    loss_history.append(loss.item() * GRAD_ACCUM)

    # Update weights every GRAD_ACCUM steps
    if (global_step + 1) % GRAD_ACCUM == 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            [p for p in unet.parameters() if p.requires_grad], max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        gc.collect(); torch.cuda.empty_cache()

    global_step += 1

    # Logging
    if global_step % LOG_EVERY == 0:
        recent = loss_history[-LOG_EVERY:]
        avg = sum(recent) / len(recent)
        elapsed = (time.time() - t0) / 60
        eta    = elapsed / global_step * (MAX_STEPS - global_step)
        vram   = torch.cuda.memory_allocated() / 1e9
        print(f'step {global_step:5d}/{MAX_STEPS}  '
              f'loss={avg:.4f}  '
              f'elapsed={elapsed:.1f}m  ETA={eta:.1f}m  '
              f'VRAM={vram:.2f}GB')

    # Checkpoint
    if global_step % SAVE_EVERY == 0:
        ckpt = f'{LORA_DIR}/step_{global_step}'
        unet.save_pretrained(ckpt)
        print(f'  Checkpoint saved -> {ckpt}')

# Final save
unet.save_pretrained(f'{LORA_DIR}/final')
print(f'\nTraining complete! Final LoRA saved -> {LORA_DIR}/final')


## 9. Plot Training Curve

Loss should decrease steadily. If it plateaus early, try raising `LR` to `2e-4` or increasing `LORA_RANK` to 8 and re-training.  
A final loss around **0.05–0.15** is typical for SDXL-based try-on fine-tuning.

In [ ]:
import matplotlib.pyplot as plt
import json, os

# Smooth with a rolling average
def smooth(vals, w=50):
    out = []
    for i in range(len(vals)):
        out.append(sum(vals[max(0,i-w):i+1]) / min(i+1, w))
    return out

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(loss_history, alpha=0.25, color='steelblue', label='raw')
ax.plot(smooth(loss_history), color='steelblue', lw=2, label='smoothed (w=50)')
ax.set_xlabel('Step'); ax.set_ylabel('MSE loss')
ax.set_title('IDM-VTON LoRA Training Loss')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{CKPT_DIR}/stage6_loss_curve.png', dpi=120)
plt.show()

# Save loss log to Drive
with open(f'{LORA_DIR}/loss_log.json', 'w') as f:
    json.dump(loss_history, f)
print(f'Loss curve saved -> {CKPT_DIR}/stage6_loss_curve.png')


## 10. Quick Inference Test with Trained LoRA

We load the saved LoRA weights back into a fresh TryonUNet, build a full pipeline, and run inference on our Stage 1-4 inputs to see whether fine-tuning improved results vs Stage 5.  

The LoRA adapter is merged into the base weights with `merge_and_unload()` for faster inference (no adapter overhead at runtime).

In [ ]:
import gc, torch
from PIL import Image
from peft import PeftModel
from diffusers import DDIMScheduler, AutoencoderKL
from transformers import (CLIPImageProcessor, CLIPVisionModelWithProjection,
                          CLIPTextModel, CLIPTextModelWithProjection, CLIPTokenizer)
import numpy as np

LORA_CKPT = f'{LORA_DIR}/final'  # or change to a specific step checkpoint

# Rebuild scheduler and pipeline components
scheduler_inf = DDIMScheduler.from_pretrained(model_path, subfolder='scheduler')

# Load base TryonUNet + merge LoRA
print('Loading TryonUNet + merging LoRA...')
_sd2 = torch.load(f'{model_path}/unet/diffusion_pytorch_model.bin', map_location=DEVICE)
for k in list(_sd2): _sd2[k] = _sd2[k].half()
gc.collect(); torch.cuda.empty_cache()
with init_empty_weights():
    unet_inf = TryonUNet.from_config(f'{model_path}/unet')
unet_inf.load_state_dict(_sd2, assign=True, strict=False)
del _sd2; gc.collect(); torch.cuda.empty_cache()

unet_inf = PeftModel.from_pretrained(unet_inf, LORA_CKPT)
unet_inf = unet_inf.merge_and_unload()  # bake LoRA into weights
print(f'TryonUNet + LoRA merged.  VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB')

# Load text encoders for prompt encoding
tok1 = CLIPTokenizer.from_pretrained(model_path, subfolder='tokenizer')
tok2 = CLIPTokenizer.from_pretrained(model_path, subfolder='tokenizer_2')
te1  = CLIPTextModel.from_pretrained(model_path, subfolder='text_encoder',
       torch_dtype=torch.float16, low_cpu_mem_usage=True).to(DEVICE)
te2  = CLIPTextModelWithProjection.from_pretrained(model_path, subfolder='text_encoder_2',
       torch_dtype=torch.float16, low_cpu_mem_usage=True).to(DEVICE)

# Build inference pipeline
pipe_inf = TryonPipeline(
    vae=vae, text_encoder=te1, text_encoder_2=te2,
    tokenizer=tok1, tokenizer_2=tok2,
    unet=unet_inf, unet_encoder=unet_encoder,
    scheduler=scheduler_inf,
    image_encoder=image_encoder, feature_extractor=CLIPImageProcessor(),
)
pipe_inf.enable_attention_slicing(1)
try: pipe_inf.enable_xformers_memory_efficient_attention()
except: pass

# Encode prompt
PROMPT = 'a person wearing a blue t-shirt, photorealistic, high quality'
NEG    = 'monochrome, lowres, bad anatomy, worst quality, low quality'
with torch.no_grad():
    pe, ne, pp, np_ = pipe_inf.encode_prompt(
        PROMPT, num_images_per_prompt=1,
        do_classifier_free_guidance=True, negative_prompt=NEG)
    ce, _, _, _ = pipe_inf.encode_prompt(
        PROMPT, num_images_per_prompt=1, do_classifier_free_guidance=False)
pipe_inf.text_encoder = pipe_inf.text_encoder_2 = None
del te1, te2; gc.collect(); torch.cuda.empty_cache()

# Load our Stage 1-4 inputs from Drive
person_img   = Image.open(f'{INPUT_DIR}/test_person.jpg').convert('RGB')
clothing_img = Image.open(f'{INPUT_DIR}/clothing.jpg').convert('RGB')
agnostic_img = Image.open(f'{OUTPUT_DIR}/agnostic_person.png').convert('RGB')
erase_mask   = Image.open(f'{OUTPUT_DIR}/erase_mask.png').convert('L')
pose_img     = Image.open(f'{OUTPUT_DIR}/pose_output.jpg').convert('RGB')

# Apply background fix: replace black bg with white for correct color
def black_to_white(img, thr=40):
    arr = np.array(img.convert('RGB'))
    bg = (arr[:,:,0]<thr) & (arr[:,:,1]<thr) & (arr[:,:,2]<thr)
    arr[bg] = [255,255,255]
    return Image.fromarray(arr)
clothing_img = black_to_white(clothing_img)

# Transform to 768x1024
def compute_transform(img, tw=TARGET_W, th=TARGET_H):
    w, h = img.size
    s = max(tw/w, th/h)
    nw, nh = int(w*s), int(h*s)
    l, u = (nw-tw)//2, (nh-th)//2
    return s, (l, u, l+tw, u+th)
def apply_t(img, s, box, mode='RGB'):
    w, h = img.size
    return img.resize((int(w*s), int(h*s)), Image.BILINEAR).crop(box).convert(mode)

s, box = compute_transform(person_img)
def tt(img): return torch.from_numpy(
    np.array(img.convert('RGB')).astype(np.float32)/127.5-1).permute(2,0,1).unsqueeze(0).half().to(DEVICE)
def tm(img): return torch.from_numpy(
    np.array(img.convert('L')).astype(np.float32)/255).unsqueeze(0).unsqueeze(0).half().to(DEVICE)

agn_r   = apply_t(agnostic_img, s, box)
mask_r  = apply_t(erase_mask,   s, box, 'L')
pose_r  = apply_t(pose_img,     s, box)
cloth_r = clothing_img.resize((TARGET_W, TARGET_H), Image.BILINEAR)

import time
print('Running inference with fine-tuned LoRA...')
t0 = time.time()
with torch.no_grad():
    out = pipe_inf(
        prompt_embeds=pe.to(DEVICE), negative_prompt_embeds=ne.to(DEVICE),
        pooled_prompt_embeds=pp.to(DEVICE), negative_pooled_prompt_embeds=np_.to(DEVICE),
        num_inference_steps=30, generator=torch.Generator(DEVICE).manual_seed(42),
        strength=1.0, pose_img=tt(pose_r), text_embeds_cloth=ce.to(DEVICE),
        cloth=tt(cloth_r), mask_image=tm(mask_r), image=tt(agn_r),
        height=TARGET_H, width=TARGET_W,
        ip_adapter_image=cloth_r,   # PIL image for CLIP preprocessing
        guidance_scale=2.0,
    )
imgs = out[0] if isinstance(out, tuple) else out.images
result_ft = imgs[0]
print(f'Done in {time.time()-t0:.1f}s')

result_ft.save(f'{OUTPUT_DIR}/stage6_result_finetuned.png')

# Compare Stage 5 (pre-trained) vs Stage 6 (fine-tuned)
import matplotlib.pyplot as plt
stage5_r = Image.open(f'{OUTPUT_DIR}/stage5_result_ours.png').convert('RGB')
fig, axes = plt.subplots(1, 4, figsize=(22, 7))
for ax, img, t in zip(axes,
        [Image.open(f'{INPUT_DIR}/test_person.jpg').convert('RGB'),
         cloth_r, stage5_r, result_ft],
        ['Person', 'Clothing', 'Stage 5 (pre-trained)', 'Stage 6 (LoRA fine-tuned)']):
    ax.imshow(img); ax.set_title(t, fontsize=13); ax.axis('off')
plt.suptitle('Stage 5 vs Stage 6', fontsize=15)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/stage6_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Comparison saved -> {OUTPUT_DIR}/stage6_comparison.png')


## Summary

**What Stage 6 achieved:**
- Applied LoRA (rank 4) to TryonUNet's attention layers (~2 M trainable params)
- Fine-tuned on VITON-HD training set with MSE denoising loss
- Stayed within T4 VRAM budget via gradient checkpointing + frozen GarmentUNet
- Saved LoRA weights (~30 MB) to Drive

**If results aren't better:**
- Try `LORA_RANK = 8` and `LR = 2e-4`
- Train for more steps (increase `MAX_STEPS` to 10000)
- Use `MAX_SAMPLES = None` to train on the full 14k dataset

**Next → Stage 7:** Wrap the fine-tuned pipeline as a FastAPI endpoint so it can be called with a person photo + clothing photo and return the try-on result.